# Otpor glatke kugle: predvidi → izračunaj → provjeri

Dimenzijska analiza daje oblik \(C_D=f(Re)\), ali ne daje samu funkciju niti jamči da je jedna korelacija valjana u svim režimima. Ovdje koristimo glatku aproksimaciju za neporemećen tok oko glatke kugle samo u rasponu \(0{,}1\le Re\le2\cdot10^5\), prije područja krize otpora.

## Predvidi

1. Zašto iz jednakog Reynoldsova broja slijedi jednak \(C_D\) samo ako su ostali uvjeti sličnosti usporedivi?
2. Hoće li linearna interpolacija po \(Re\) i log–log interpolacija dati isti rezultat između rijetkih tabličnih točaka?
3. Što treba učiniti kada je traženi \(Re\) izvan kalibriranog raspona: ekstrapolirati bez upozorenja ili zaustaviti račun?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RE_MIN, RE_MAX = 0.1, 2.0e5

def cd_correlation(Re):
    # Didaktička aproksimacija glatke kugle unutar eksplicitnog raspona.
    Re = np.asarray(Re, dtype=float)
    if np.any((Re < RE_MIN) | (Re > RE_MAX)):
        raise ValueError(f"Model nije dopušten izvan {RE_MIN:g} ≤ Re ≤ {RE_MAX:g}.")
    return 24/Re*(1+0.15*Re**0.687) + 0.42/(1+4.25e4*Re**(-1.16))

def loglog_interpolate(Re, re_table, cd_table):
    Re = np.asarray(Re, dtype=float)
    if np.any((Re < re_table[0]) | (Re > re_table[-1])):
        raise ValueError("Interpolacija nije ekstrapolacija: Re je izvan tablice.")
    return np.exp(np.interp(np.log(Re), np.log(re_table), np.log(cd_table)))

re_nodes = np.geomspace(RE_MIN, RE_MAX, 14)
cd_nodes = cd_correlation(re_nodes)
re_dense = np.geomspace(RE_MIN, RE_MAX, 800)
cd_true = cd_correlation(re_dense)
cd_log = loglog_interpolate(re_dense, re_nodes, cd_nodes)
cd_linear = np.interp(re_dense, re_nodes, cd_nodes)

relative_log_error = np.abs(cd_log/cd_true-1)
relative_method_difference = np.abs(cd_linear/cd_log-1)
print(f"Najveća pogreška rijetke log–log tablice: {100*relative_log_error.max():.2f} %")
print(f"Najveća razlika linearne i log–log interpolacije: {100*relative_method_difference.max():.2f} %")


## Izračunaj: radna točka i interpolacijska osjetljivost

Za zrak računamo \(Re\), interpolirani \(C_D\) i silu. Zatim tablicu prorjeđujemo i zgušnjavamo. Promjena rezultata je procjena **interpolacijske**, a ne eksperimentalne ni modelske nesigurnosti.


In [ ]:
rho, nu, velocity, diameter = 1.20, 1.50e-5, 0.60, 0.030
Re_work = velocity*diameter/nu
Cd_work = float(loglog_interpolate(Re_work, re_nodes, cd_nodes))
area = np.pi*diameter**2/4
drag = 0.5*rho*velocity**2*Cd_work*area

node_counts = np.array([8, 12, 18, 30, 50])
cd_by_resolution = []
for count in node_counts:
    re_tab = np.geomspace(RE_MIN, RE_MAX, int(count))
    cd_tab = cd_correlation(re_tab)
    cd_by_resolution.append(float(loglog_interpolate(Re_work, re_tab, cd_tab)))
cd_by_resolution = np.asarray(cd_by_resolution)

print(f"Radna točka: Re={Re_work:.0f}, Cd={Cd_work:.4f}, F_D={1e3*drag:.4f} mN")
print("Cd pri broju tabličnih točaka:", dict(zip(node_counts, np.round(cd_by_resolution, 5))))


## Provjeri

Provjeravamo reprodukciju poznatih tabličnih čvorova, približavanje Stokesovu graničnom zakonu pri malom \(Re\) i obvezno zaustavljanje izvan raspona. Posljednja provjera sprječava prividno preciznu ekstrapolaciju kroz krizu otpora, gdje hrapavost i turbulencija slobodnog toka postaju bitne.


In [ ]:
assert np.allclose(loglog_interpolate(re_nodes, re_nodes, cd_nodes), cd_nodes, rtol=1e-13)
assert abs(float(cd_correlation(RE_MIN))/(24/RE_MIN)-1) < 0.04
outside_blocked = False
try:
    cd_correlation(5e5)
except ValueError:
    outside_blocked = True
assert outside_blocked
assert drag > 0 and RE_MIN <= Re_work <= RE_MAX

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(re_dense, cd_true, color="#256d85", lw=2, label="korelacija")
axes[0].loglog(re_nodes, cd_nodes, "o", color="#b43c35", label="rijetka tablica")
axes[0].loglog(re_dense, cd_log, "--", color="#d28f2c", label="log–log interpolacija")
axes[0].set(xlabel="Re", ylabel="$C_D$", title="Model samo u označenom rasponu")
axes[0].legend(fontsize=8)
axes[1].semilogx(re_dense, 100*relative_method_difference, color="#7a3e9d")
axes[1].set(xlabel="Re", ylabel="razlika metoda (%)", title="Osjetljivost na interpolaciju")
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Veći broj tabličnih točaka smanjuje samo numeričku interpolacijsku pogrešku. Ne uklanja pogrešku korelacije niti nadomješta mjerenja za hrapavu kuglu, blizinu stijenke ili turbulentni slobodni tok.


## Usporedi s primjerom P5

Predvidi jesu li zadani $C_D=0{,}45$ i rezultat korelacije pri $Re=40000$ nužno jednaki. Zadani podatak iz P5 i didaktička korelacija dva su različita ulaza. Razlika nije interpolacijska pogreška i ne smije se sakriti pomicanjem radne točke na krivulju.

In [ ]:
re_p5, cd_given = 4.0e4, 0.45
cd_model = float(cd_correlation(re_p5))
force_given = 0.5*1.20*30.0**2*(np.pi*0.020**2/4)*cd_given
assert np.isclose(force_given, 0.076340701482232, rtol=1e-12)
print(f"P5: zadani Cd={cd_given:.3f}; korelacija Cd={cd_model:.4f}; sila iz zadanog Cd={1e3*force_given:.2f} mN")

## Z4: prijenos otpora hidroprofila

**Predvidi:** povećava li zrak potrebnu modelsku brzinu? Može li se ovdje sila prenijeti faktorom $λ_L^3$?

**Izračunaj i provjeri:** uzmi iste pretpostavke Z4: geometrijska sličnost, jednak relativni dovodni režim i hrapavost, nulti napadni kut, zanemarivi utjecaji stijenki, slobodne površine i kavitacije. Zadržavamo $Re$ i provjeravamo mali $Ma_m$. Referentna površina je $bc$; sintetičko očitanje jest komponenta otpora u smjeru struje.

In [ ]:
cp, bp, vp, rhop, nup = 0.300, 0.600, 1.00, 1000., 1.00e-6
cm, bm, rhom, num, am, fdm = 0.100, 0.200, 1.20, 1.50e-5, 340., 0.405
rep = vp*cp/nup
vm = rep*num/cm
mam = vm/am
cd_foil = fdm/(0.5*rhom*vm**2*bm*cm)
fdp = 0.5*rhop*vp**2*bp*cp*cd_foil
assert np.isclose(vm*cm/num, rep, rtol=1e-13)
assert np.isclose(vm, 45.) and mam < 0.3
assert np.isclose(fdp, 1.500, rtol=1e-13)
assert np.isclose(fdp/fdm, rhop/rhom*(vp/vm)**2*(bp*cp)/(bm*cm))
print(f"Re={rep:.0f}; vm={vm:.2f} m/s; Ma={mam:.4f}; Cd={cd_foil:.6f}; Fp={fdp:.3f} N")

## Z6: mjerilo, intervali i izvedivost pokusa

**Predvidi:** koje mjerilo traži veći protok, a daje veću mjerenu silu? Zašto preklapanje dvaju prenesenih intervala ne uklanja pogrešku zbog različitih Reynoldsovih brojeva?

Podatci su sintetički. Zajamčene granice pogreške prenose se s oba kraja intervala; nisu standardne nesigurnosti za statističko zbrajanje. Faktor $λ_L^3$ ovdje je uvjetna procjena iz Froudeove sličnosti uz jednaku gustoću i gravitaciju.

In [ ]:
scale = np.array([20., 30.])
fm, dfm = np.array([28., 8.]), np.array([0.6, 0.2])
vp, hp, qp, nu, g = 6., 7.5, 480., 1.0e-6, 9.81
hm = hp/scale
vm = vp/np.sqrt(scale)
qm = qp/scale**2.5
bm = (qp/(vp*hp))/scale
rem = vm*hm/nu
fp, dfp = scale**3*fm/1000., scale**3*dfm/1000.
lower, upper = fp-dfp, fp+dfp
intersection = np.array([lower.max(), upper.min()])
eligible = (qm <= 0.300) & (fm-dfm > 10.)
assert np.allclose(vm/np.sqrt(g*hm), vp/np.sqrt(g*hp), rtol=1e-13)
assert np.allclose(qm, vm*bm*hm, rtol=1e-13)
assert np.allclose((vp*hp/nu)/rem, scale**1.5, rtol=1e-13)
assert np.allclose(intersection, [219.2, 221.4], atol=1e-10)
assert np.array_equal(eligible, [True, False])
for i, lam in enumerate(scale):
    print(f"1:{lam:g}: vm={vm[i]:.4f} m/s; Qm={1e3*qm[i]:.2f} L/s; Re={rem[i]:.0f}; Fp=[{lower[i]:.1f}, {upper[i]:.1f}] kN; izvedivo={eligible[i]}")
fig, ax = plt.subplots(figsize=(7, 3))
ax.errorbar(fp, scale, xerr=dfp, fmt='o', capsize=6, color='#1565c0')
ax.axvspan(*intersection, color='#1e8449', alpha=.15, label='presjek intervala')
ax.set(xlabel='Uvjetno prenesena sila na prototip (kN)', ylabel='Mjerilo λL', yticks=scale,
       title='Sintetički podatci: zajamčene granice mjerenja')
ax.legend(); ax.grid(axis='x', alpha=.3); plt.tight_layout(); plt.show()

## Protumači rezultate prijenosa

1. Usporedi faktor prijenosa sile hidroprofila s $(c_p/c_m)^3$. Koja pretpostavka za potonji faktor ovdje izostaje?
2. Objasni zašto jednak $Re$ nije dovoljan ako se promijene relativna hrapavost ili bezdimenzijski rubni uvjeti.
3. Koji model preljeva isključuje mjerno ograničenje, iako zadovoljava ograničenje protoka?
4. Kad bi zajamčene granice sile bile prepolovljene, bi li intervali još imali presjek? To provjeri i odvoji neslaganje očitanja od dokaza uzroka pogreške.
5. Navedi barem dvije dodatne provjere potrebne prije tvrdnje da Froudeov model daje malu pogrešku prijenosa sile.